# CEGA-Tiny v3: Confidence-Gated, Energy-Governed On-Device Learning for TinyML (submission experiments)

**Self-contained experiment notebook for Google Colab** (CPU runtime is enough; no GPU needed).

What this notebook does, end to end:

1. Downloads four public drifting streams: **UCI Gas Sensor Array Drift**, **UCI-HAR** (ordered by subject),
   **Elec2** (OpenML 151) and **Forest CoverType** (first 60,000 records, via scikit-learn).
2. Pre-trains a tiny MLP on the first part of each stream, then streams the rest in windows (prequential
   test-then-train) with sparse labels (20 %), a FIFO replay buffer and a simulated energy-harvesting battery.
3. **Main experiment:** CEGA-Tiny against 7 baselines, including a **RandomSkip** control that skips the same
   fraction of samples at random, plus 6 ablations including the minimal **CEGA-Lite**, over 5 seeds.
4. **Sensitivity and robustness sweeps** (3 seeds each): energy budget, label rate, label noise,
   gate thresholds, energy-constant ratio and model size.
5. Statistics (Friedman; Wilcoxon with Holm correction; 95 % CIs; effect sizes), CSV and LaTeX tables, and
   300-DPI figures (PNG and PDF).
6. **Optional hardware calibration:** after running `cega_tiny_bench.ino` on a microcontroller, enter the
   measured times and currents in section 12 to refit the energy constants and repeat the Gas Drift comparison.

**CEGA-Tiny in one paragraph.** The device already runs a forward pass for every sample, so the probability it
gives the true class of each newly labelled sample is free. Only samples whose true-class probability is below
a threshold tau are back-propagated, and tau rises with the battery state of charge. Below a low-SoC limit only
the classifier head is updated. Page-Hinkley alarms switch gating off for a few windows; an error alarm also
triggers one replay epoch over the buffer if the battery can afford it.

**Note on energy values:** energy is estimated from MAC and parameter-write counts with the constants in `ENERGY`.
These are assumptions unless you complete the hardware calibration in section 12.

Run: `Runtime -> Run all`. Expected time on a standard Colab CPU: roughly 30-60 minutes.

In [2]:
# ============================== 0. Imports & configuration ==============================
import os, io, sys, json, time, math, zipfile, shutil, warnings, urllib.request
from collections import deque
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg") if "google.colab" not in sys.modules else None
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")

SMOKE = os.environ.get("CEGA_SMOKE", "0") == "1"   # quick synthetic self-test only; keep False on Colab
OUT = "cega_tiny_results"
DATA_DIR = "data"
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
for sub in ["figures", "tables", "raw"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)

CFG = dict(
    seeds=[0, 1, 2, 3, 4],
    window=50,            # samples per stream window
    p_label=0.20,         # fraction of stream samples whose label is revealed (sparse supervision)
    buffer=200,           # FIFO replay buffer capacity (labelled samples)
    min_buffer=40,        # minimum labelled samples before any retraining
    hidden=(64, 32),      # tiny MLP hidden sizes
    pre_epochs=40, pre_lr=0.05,
    adapt_epochs=5,       # fixed epochs used by retraining baselines
    max_epochs=8,         # upper bound for CEGA (early stopping may use fewer)
    lr_adapt=0.02, lr_online=0.01, momentum=0.9, batch=32,
    beta=0.5,             # spare harvested energy per window = beta * E_full / 20
    cap_mult=3.0,         # battery capacity = cap_mult * E_full
    soc0=0.5,             # initial state of charge
    reserve=0.10,         # CEGA never plans below this SoC
    period=20,            # Periodic-Full retraining period (windows)
    lambda0=0.3,          # CEGA energy price
    g_scale=0.10,         # converts normalised energy into accuracy units
    g_prior={1: 0.04, 2: 0.06, 3: 0.08},  # prior expected accuracy gain per adaptation depth
    g_ema=0.3,
    pending_max=10, cooldown=3, recent=3,
    ph_err=dict(delta=0.005, lam=15.0, min_n=30),
    ph_ent=dict(delta=0.01, lam=0.30, min_n=5),
    # ---- CEGA-Tiny v2 (energy-governed gating of on-device online learning) ----
    v2_tau_min=0.50, v2_tau_max=0.95,   # update a labelled sample only if p(true class) < tau; tau grows with SoC
    v2_boost=5,                         # windows of ungated full-depth updates after a detected drift
    v2_burst_epochs=1,                  # replay epochs over the buffer when drift is detected
    v2_low_soc=0.25,                    # below this SoC only the classifier head is updated
    label_noise=0.0,                    # fraction of revealed labels that are wrong (robustness study)
    beta_sweep=[0.1, 0.25, 0.5, 1.0, 2.0],
    rho_sweep=[0.05, 0.10, 0.20, 0.50],          # label-availability sensitivity
    noise_sweep=[0.0, 0.10, 0.20, 0.30],         # label-noise robustness
    tau_min_grid=[0.3, 0.5, 0.7], tau_max_grid=[0.80, 0.90, 0.95, 0.99],
    ewr_sweep=[0.25, 1.0, 4.0, 16.0],            # nJ per parameter write (e_mac fixed) -> write/MAC cost ratio 1..64
    hidden_sweep=[(32, 16), (64, 32), (128, 64)],
    sweep_seeds=[0, 1, 2],
)
# Per-operation energy model (assumptions; see the note on energy values in the title cell). nJ per multiply-accumulate and per parameter write.
ENERGY = dict(e_mac_nj=0.25, e_write_nj=1.0)

if SMOKE:
    CFG.update(seeds=[0, 1], pre_epochs=10, beta_sweep=[0.25, 1.0], sweep_seeds=[0], rho_sweep=[0.1, 0.2],
               noise_sweep=[0.0, 0.2], tau_min_grid=[0.5], tau_max_grid=[0.9, 0.95], ewr_sweep=[1.0, 16.0],
               hidden_sweep=[(32, 16), (64, 32)])

MAIN_METHODS = ["Static", "Online-SGD", "TinyOL-Head", "RandomSkip", "Periodic-Full", "DriftTrig-Full", "GovRetrain",
                "CEGA-Tiny"]
# GovRetrain = energy-governed *buffer retraining* (our first design, v1); kept as a baseline.
# RandomSkip = control: skips labelled samples at random with the SAME update rate CEGA-Tiny used (same dataset/seed).
ABLATIONS = ["CEGA-Lite", "CEGA-noGate", "CEGA-noBurst", "CEGA-noSoC", "CEGA-noEntropy", "CEGA-HeadOnly"]
# CEGA-Lite = minimal configuration: gate + SoC-scaled threshold + head-only fallback (no replay, no entropy alarm).
ALL_METHODS = MAIN_METHODS + ABLATIONS
SWEEP_METHODS = ["Online-SGD", "Periodic-Full", "DriftTrig-Full", "CEGA-Tiny"]
print("Config ready. SMOKE =", SMOKE)

Config ready. SMOKE = False


In [3]:
# ============================== 1. Data loading ==============================
UA = {"User-Agent": "Mozilla/5.0 (research script)"}

def download(url, dest, tries=3):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return dest
    last = None
    for t in range(tries):
        try:
            req = urllib.request.Request(url, headers=UA)
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
                shutil.copyfileobj(r, f)
            return dest
        except Exception as e:
            last = e; time.sleep(3 * (t + 1))
    raise RuntimeError(f"Download failed for {url}: {last}")

def _parse_gas_lines(text):
    X, y = [], []
    for line in text.splitlines():
        tok = line.strip().split()
        if not tok:
            continue
        y.append(int(tok[0].split(";")[0]) - 1)          # label format: "class;concentration"
        row = np.zeros(128, np.float32)
        for kv in tok[1:]:
            k, v = kv.split(":"); row[int(k) - 1] = float(v)
        X.append(row)
    return np.array(X), np.array(y)

def load_gas(seed):
    """UCI Gas Sensor Array Drift (Vergara et al., 2012). Pre-train on batches 1-2; stream batches 3-10."""
    p = download("https://archive.ics.uci.edu/static/public/224/gas+sensor+array+drift+dataset.zip",
                 os.path.join(DATA_DIR, "gas.zip"))
    Xs, ys, bs = [], [], []
    with zipfile.ZipFile(p) as z:
        names = {os.path.basename(n): n for n in z.namelist()}
        for b in range(1, 11):
            X, y = _parse_gas_lines(z.read(names[f"batch{b}.dat"]).decode())
            Xs.append(X); ys.append(y); bs.append(np.full(len(y), b))
    rng = np.random.default_rng(1000 + seed)
    Xp = np.vstack(Xs[:2]); yp = np.concatenate(ys[:2])
    Xstream, ystream, drift = [], [], []
    pos = 0
    for b in range(2, 10):
        perm = rng.permutation(len(ys[b]))                 # order inside a batch is unknown -> shuffle per seed
        Xstream.append(Xs[b][perm]); ystream.append(ys[b][perm])
        if b > 2: drift.append(pos)
        pos += len(perm)
    return dict(name="GasDrift", Xp=Xp, yp=yp, Xs=np.vstack(Xstream), ys=np.concatenate(ystream),
                drift=drift, C=6)

def load_har(seed):
    """UCI HAR (Anguita et al., 2013). Stream ordered by subject; pre-train on the first 6 subjects."""
    p = download("https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
                 os.path.join(DATA_DIR, "har.zip"))
    with zipfile.ZipFile(p) as z:
        inner = [n for n in z.namelist() if n.endswith("UCI HAR Dataset.zip")]
        zz = zipfile.ZipFile(io.BytesIO(z.read(inner[0]))) if inner else z
        def rd(part, kind):
            n = [m for m in zz.namelist() if m.endswith(f"{part}/{kind}_{part}.txt")][0]
            return pd.read_csv(io.BytesIO(zz.read(n)), sep=r"\s+", header=None).values
        X = np.vstack([rd("train", "X"), rd("test", "X")]).astype(np.float32)
        y = np.concatenate([rd("train", "y").ravel(), rd("test", "y").ravel()]) - 1
        s = np.concatenate([rd("train", "subject").ravel(), rd("test", "subject").ravel()])
    order = np.argsort(s, kind="stable")
    X, y, s = X[order], y[order], s[order]
    subj = np.unique(s); pre = np.isin(s, subj[:6])
    Xs, ys, ss = X[~pre], y[~pre], s[~pre]
    drift = [i for i in range(1, len(ss)) if ss[i] != ss[i - 1]]
    return dict(name="UCI-HAR", Xp=X[pre], yp=y[pre], Xs=Xs, ys=ys, drift=drift, C=6)

def load_elec(seed):
    """Electricity / Elec2 (Harries, 1999) from OpenML id 151. Chronological; pre-train on first 4000."""
    from sklearn.datasets import fetch_openml
    d = fetch_openml(data_id=151, as_frame=True, parser="auto", data_home=DATA_DIR)
    df = d.data.copy()
    df = df.drop(columns=[c for c in df.columns if c.lower() == "date"])
    X = df.apply(pd.to_numeric, errors="coerce").fillna(0).values.astype(np.float32)
    y = (d.target.astype(str).str.upper() == "UP").astype(int).values
    return dict(name="Elec2", Xp=X[:4000], yp=y[:4000], Xs=X[4000:], ys=y[4000:], drift=[], C=2)

def make_synthetic(name, seed, d=20, C=4, n_pre=1500, n_seg=1200, segs=5):
    """Smoke-test only: Gaussian classes whose means move at each segment. Never used for reported results."""
    rng = np.random.default_rng(seed + 7)
    mu = rng.normal(0, 2, (C, d))
    def seg(n, mu):
        y = rng.integers(0, C, n); return (mu[y] + rng.normal(0, 1, (n, d))).astype(np.float32), y
    Xp, yp = seg(n_pre, mu); Xs, ys, drift = [], [], []
    for k in range(segs):
        if k: mu = mu + rng.normal(0, 1.2, mu.shape); drift.append(k * n_seg)
        X, y = seg(n_seg, mu); Xs.append(X); ys.append(y)
    return dict(name=name, Xp=Xp, yp=yp, Xs=np.vstack(Xs), ys=np.concatenate(ys), drift=drift, C=C)

def load_covtype(seed):
    """Forest CoverType (Blackard & Dean, 1999), a standard real-world stream benchmark. First 60,000 records in
    original order; pre-train on the first 5,000."""
    from sklearn.datasets import fetch_covtype
    d = fetch_covtype(data_home=DATA_DIR)
    X = d.data[:60000].astype(np.float32); y = d.target[:60000].astype(int) - 1
    return dict(name="CoverType", Xp=X[:5000], yp=y[:5000], Xs=X[5000:], ys=y[5000:], drift=[], C=7)

LOADERS = {"GasDrift": load_gas, "UCI-HAR": load_har, "Elec2": load_elec, "CoverType": load_covtype}

def prepare(ds):
    """Standardise with pre-deployment statistics only (the device never sees future data)."""
    mu = ds["Xp"].mean(0); sd = ds["Xp"].std(0) + 1e-6
    f = lambda X: np.clip((X - mu) / sd, -8, 8).astype(np.float32)
    ds["Xp"], ds["Xs"] = f(ds["Xp"]), f(ds["Xs"])
    return ds

_DS_CACHE = {}
def get_dataset(name, seed):
    if (name, seed) not in _DS_CACHE:
        ds = make_synthetic(name, seed) if SMOKE else LOADERS[name](seed)
        _DS_CACHE[(name, seed)] = prepare(ds)
    return _DS_CACHE[(name, seed)]

DATASETS = ["GasDrift", "UCI-HAR", "Elec2", "CoverType"]
for n in DATASETS:
    d = get_dataset(n, 0)
    print(f"{n:9s} pre={len(d['yp']):6d} stream={len(d['ys']):6d} feat={d['Xs'].shape[1]:4d} "
          f"classes={d['C']} known drift points={len(d['drift'])}")

GasDrift  pre=  1689 stream= 12221 feat= 128 classes=6 known drift points=7
UCI-HAR   pre=  1934 stream=  8365 feat= 561 classes=6 known drift points=23
Elec2     pre=  4000 stream= 41312 feat=   7 classes=2 known drift points=0
CoverType pre=  5000 stream= 55000 feat=  54 classes=7 known drift points=0


In [4]:
# ============================== 2. Tiny MLP with exact op counting ==============================
class TinyMLP:
    def __init__(self, sizes, rng):
        self.sizes = list(sizes); self.L = len(sizes) - 1
        self.W = [rng.normal(0, np.sqrt(2 / i), (i, o)).astype(np.float32) for i, o in zip(sizes[:-1], sizes[1:])]
        self.b = [np.zeros(o, np.float32) for o in sizes[1:]]

    def copy(self):
        m = TinyMLP.__new__(TinyMLP); m.sizes = list(self.sizes); m.L = self.L
        m.W = [w.copy() for w in self.W]; m.b = [b.copy() for b in self.b]; return m

    def forward(self, X):
        acts, h = [X], X
        for l in range(self.L):
            z = h @ self.W[l] + self.b[l]
            h = np.maximum(z, 0) if l < self.L - 1 else z
            acts.append(h)
        return acts

    def proba(self, X):
        z = self.forward(X)[-1]; z = z - z.max(1, keepdims=True); e = np.exp(z)
        return e / e.sum(1, keepdims=True)

    def layer_macs(self):
        return [i * o for i, o in zip(self.sizes[:-1], self.sizes[1:])]

    def macs_fwd(self):
        return sum(self.layer_macs())

    def macs_train_sample(self, depth):
        lm = self.layer_macs(); top = lm[self.L - depth:]
        return self.macs_fwd() + sum(top) + sum(top[1:])   # forward + weight-grads + input-grads above the lowest trained layer

    def n_params(self, depth=None):
        depth = self.L if depth is None else depth
        return sum(self.W[l].size + self.b[l].size for l in range(self.L - depth, self.L))

    def grads(self, X, y, depth):
        acts = self.forward(X); n = len(X)
        z = acts[-1] - acts[-1].max(1, keepdims=True); p = np.exp(z); p /= p.sum(1, keepdims=True)
        loss = -np.log(p[np.arange(n), y] + 1e-9).mean()
        d = p; d[np.arange(n), y] -= 1; d /= n
        gW, gb = {}, {}
        for l in range(self.L - 1, self.L - depth - 1, -1):
            gW[l] = acts[l].T @ d; gb[l] = d.sum(0)
            if l > self.L - depth:
                d = (d @ self.W[l].T) * (acts[l] > 0)
        return gW, gb, loss

def loss_acc(model, X, y):
    p = model.proba(X)
    return float(-np.log(p[np.arange(len(y)), y] + 1e-9).mean()), float((p.argmax(1) == y).mean())

def sgd_train(model, X, y, depth, epochs, lr, rng, mom=0.9, batch=32, Xv=None, yv=None, early_stop=False, patience=1):
    """SGD with momentum on the top `depth` layers. Returns op counts for energy costing."""
    vel = {l: (np.zeros_like(model.W[l]), np.zeros_like(model.b[l])) for l in range(model.L - depth, model.L)}
    steps = samples = val_evals = 0; ep_run = 0
    best, bad, best_state = np.inf, 0, None
    for ep in range(epochs):
        perm = rng.permutation(len(X))
        for s in range(0, len(X), batch):
            idx = perm[s:s + batch]
            gW, gb, _ = model.grads(X[idx], y[idx], depth)
            gn = math.sqrt(sum(float((g ** 2).sum()) for g in gW.values()))
            sc = min(1.0, 5.0 / (gn + 1e-12))
            for l in gW:
                vW, vb = vel[l]
                vW *= mom; vW -= lr * sc * gW[l]; vb *= mom; vb -= lr * sc * gb[l]
                model.W[l] += vW; model.b[l] += vb
            steps += 1; samples += len(idx)
        ep_run += 1
        if early_stop and Xv is not None and len(yv):
            vl, _ = loss_acc(model, Xv, yv); val_evals += len(yv)
            if vl < best - 1e-4:
                best, bad = vl, 0
                best_state = [(model.W[l].copy(), model.b[l].copy()) for l in range(model.L - depth, model.L)]
            else:
                bad += 1
                if bad > patience: break
    if early_stop and best_state is not None:
        for k, l in enumerate(range(model.L - depth, model.L)):
            model.W[l], model.b[l] = best_state[k]
    return dict(steps=steps, samples=samples, val_evals=val_evals, epochs=ep_run)

def train_energy_mJ(model, depth, ops):
    e = ops["samples"] * model.macs_train_sample(depth) * ENERGY["e_mac_nj"]
    e += ops["val_evals"] * model.macs_fwd() * ENERGY["e_mac_nj"]
    e += ops["steps"] * model.n_params(depth) * ENERGY["e_write_nj"]
    return e * 1e-6   # nJ -> mJ

def planned_energy_mJ(model, depth, n, epochs, n_val=0):
    steps = epochs * math.ceil(n / CFG["batch"])
    return train_energy_mJ(model, depth, dict(samples=epochs * n, steps=steps, val_evals=epochs * n_val))

In [5]:
# ============================== 3. Drift detection ==============================
class PageHinkley:
    """Page-Hinkley test for an increase in the mean of a signal."""
    def __init__(self, delta, lam, min_n):
        self.delta, self.lam, self.min_n = delta, lam, min_n; self.reset()
    def reset(self):
        self.n, self.mean, self.cum, self.min = 0, 0.0, 0.0, 0.0
    def update(self, x):
        self.n += 1; self.mean += (x - self.mean) / self.n
        self.cum += x - self.mean - self.delta; self.min = min(self.min, self.cum)
        return self.n >= self.min_n and (self.cum - self.min) > self.lam

def norm_entropy(P):
    return float((-(P * np.log(P + 1e-12)).sum(1) / np.log(P.shape[1])).mean())

In [6]:
# ============================== 4. Streaming simulator (all methods) ==============================
PRETRAINED = {}

def pretrained_model(ds, seed):
    key = (ds["name"], seed, tuple(CFG["hidden"]))
    if key not in PRETRAINED:
        rng = np.random.default_rng(seed)
        n = len(ds["yp"]); perm = rng.permutation(n); nv = int(0.15 * n)
        tr, va = perm[nv:], perm[:nv]
        m = TinyMLP([ds["Xs"].shape[1], *CFG["hidden"], ds["C"]], rng)
        sgd_train(m, ds["Xp"][tr], ds["yp"][tr], m.L, CFG["pre_epochs"], CFG["pre_lr"], rng, batch=64)
        _, acc = loss_acc(m, ds["Xp"][va], ds["yp"][va])
        P = m.proba(ds["Xp"][va])
        PRETRAINED[key] = (m, 1 - acc, norm_entropy(P))
    return PRETRAINED[key]

def recovery_windows(win_acc, drift_samples, W, cap=20, pre=5):
    """Windows needed after a known drift point for the 3-window mean accuracy to regain 95% of the pre-drift level."""
    wa = np.asarray(win_acc); out = []
    dws = sorted(set(d // W for d in drift_samples))
    for i, dw in enumerate(dws):
        if dw < pre or dw >= len(wa) - 3: continue
        nxt = dws[i + 1] if i + 1 < len(dws) else len(wa)
        ref = wa[dw - pre:dw].mean(); r = cap
        for k in range(dw, min(dw + cap, nxt, len(wa) - 2)):
            if wa[k:k + 3].mean() >= 0.95 * ref: r = k - dw; break
        out.append(r)
    return float(np.mean(out)) if out else np.nan

def run_stream(ds, method, seed, beta=None, keep_trace=False, keep=None):
    beta = CFG["beta"] if beta is None else beta
    base, pre_err, pre_ent = pretrained_model(ds, seed)
    model = base.copy(); rng = np.random.default_rng(10_000 + seed)
    Xs, ys, W, B = ds["Xs"], ds["ys"], CFG["window"], CFG["buffer"]
    nwin = len(ys) // W; n = nwin * W
    lab = np.random.default_rng(20_000 + seed).random(n) < CFG["p_label"]   # same label mask for every method
    # labels the device observes (optionally corrupted); evaluation always uses the true labels ys
    yo = ys.copy()
    if CFG.get("label_noise", 0.0) > 0:
        nr = np.random.default_rng(30_000 + seed)
        idx = np.where(lab & (nr.random(n) < CFG["label_noise"]))[0]
        yo[idx] = (ys[idx] + nr.integers(1, ds["C"], len(idx))) % ds["C"]
    rs_rng = np.random.default_rng(40_000 + seed)
    buf = deque(maxlen=B)

    L = model.L
    E_full = planned_energy_mJ(model, L, B, CFG["max_epochs"])
    cap = CFG["cap_mult"] * E_full; batt = CFG["soc0"] * cap
    E_inf_win = W * model.macs_fwd() * ENERGY["e_mac_nj"] * 1e-6
    harvest = E_inf_win + beta * E_full / 20.0

    # method switches
    is_v2 = method.startswith("CEGA")          # proposed CEGA-Tiny and its ablations
    is_cega = method == "GovRetrain"           # v1 design: energy-governed buffer retraining
    v2_ent = method not in ("CEGA-noEntropy", "CEGA-Lite"); v2_gate = method != "CEGA-noGate"
    v2_burst = method not in ("CEGA-noBurst", "CEGA-Lite"); v2_soc = method != "CEGA-noSoC"; v2_head = method == "CEGA-HeadOnly"
    acts = dict(skip=0, head=0, full=0, burst=0)
    n_lab = n_upd = 0
    boost = 0
    use_ent = calib = use_soc = use_es = True; depths = [1, 2, 3]
    g = dict(CFG["g_prior"])

    ph_err = PageHinkley(**CFG["ph_err"]); ph_ent = PageHinkley(**CFG["ph_ent"])
    preds = np.empty(n, int); win_acc, soc_tr, events = [], [], []
    hist_err, hist_n, hist_ent = [], [], []          # per-window labelled errors / counts / entropy since last reset
    E_inf = E_ad = 0.0; n_ad = viol = 0; deficit = 0.0; min_soc = 1.0
    pending = cool = 0; depth_count = {1: 0, 2: 0, 3: 0}

    def buf_xy():
        idx = np.fromiter(buf, int); return Xs[idx], yo[idx]

    for w in range(nwin):
        sl = slice(w * W, (w + 1) * W); Xw, yw, ywo = Xs[sl], ys[sl], yo[sl]
        P = model.proba(Xw); pw = P.argmax(1); preds[sl] = pw
        win_acc.append(float((pw == yw).mean()))
        batt = min(cap, batt + harvest) - E_inf_win; E_inf += E_inf_win
        li = np.where(lab[sl])[0]
        for i in li: buf.append(w * W + i)
        errs = (pw[li] != ywo[li]).astype(float); H = norm_entropy(P); n_lab += len(li)
        hist_err.append(errs.sum()); hist_n.append(len(li)); hist_ent.append(H)
        E_step = 0.0; cool = max(0, cool - 1)

        if method in ("Online-SGD", "TinyOL-Head") and len(li):
            d = L if method == "Online-SGD" else 1
            ops = sgd_train(model, Xw[li], ywo[li], d, 1, CFG["lr_online"], rng, CFG["momentum"], CFG["batch"])
            E_step = train_energy_mJ(model, d, ops); n_ad += 1; n_upd += len(li)

        elif method == "RandomSkip" and len(li):
            # control: keeps each labelled sample with probability `keep` (matched to CEGA-Tiny's update rate)
            sel = li[rs_rng.random(len(li)) < keep]
            if len(sel):
                ops = sgd_train(model, Xw[sel], ywo[sel], L, 1, CFG["lr_online"], rng, CFG["momentum"], CFG["batch"])
                E_step = train_energy_mJ(model, L, ops); n_ad += 1; n_upd += len(sel)

        elif method == "Periodic-Full":
            if (w + 1) % CFG["period"] == 0 and len(buf) >= CFG["min_buffer"]:
                Xb, yb = buf_xy()
                ops = sgd_train(model, Xb, yb, L, CFG["adapt_epochs"], CFG["lr_adapt"], rng, CFG["momentum"], CFG["batch"])
                E_step = train_energy_mJ(model, L, ops); n_ad += 1; events.append((w, L))

        elif method == "DriftTrig-Full":
            fired = any([ph_err.update(e) for e in errs])
            if fired and cool == 0 and len(buf) >= CFG["min_buffer"]:
                Xb, yb = buf_xy()
                ops = sgd_train(model, Xb, yb, L, CFG["adapt_epochs"], CFG["lr_adapt"], rng, CFG["momentum"], CFG["batch"])
                E_step = train_energy_mJ(model, L, ops); n_ad += 1; events.append((w, L))
                ph_err.reset(); cool = CFG["cooldown"]

        elif is_v2:
            fired_err = any([ph_err.update(e) for e in errs])
            fired_ent = ph_ent.update(H) if v2_ent else False
            if (fired_err or fired_ent) and cool == 0:
                # entropy alarm (label-free) -> ungated boost only; error alarm (label evidence) -> boost + replay burst
                boost = CFG["v2_boost"]; cool = CFG["cooldown"]; ph_err.reset(); ph_ent.reset()
                if fired_err and v2_burst and len(buf) >= CFG["min_buffer"]:
                    Ea = planned_energy_mJ(model, L, len(buf), CFG["v2_burst_epochs"])
                    if batt - Ea >= CFG["reserve"] * cap:
                        Xb, yb = buf_xy()
                        ops = sgd_train(model, Xb, yb, L, CFG["v2_burst_epochs"], CFG["lr_adapt"], rng, CFG["momentum"], CFG["batch"])
                        E_step += train_energy_mJ(model, L, ops); n_ad += 1; acts["burst"] += 1; events.append((w, L))
            if len(li):
                soc = max(batt - E_step, 0.0) / cap
                if boost > 0 or not v2_gate:
                    tau = 1.01
                else:
                    tau = CFG["v2_tau_min"] + (CFG["v2_tau_max"] - CFG["v2_tau_min"]) * (min(soc, 1.0) if v2_soc else 0.5)
                boost = max(0, boost - 1)
                sel = li[P[li, ywo[li]] < tau]
                if not len(sel): acts["skip"] += 1
                if len(sel):
                    d = 1 if (v2_head or (v2_soc and soc < CFG["v2_low_soc"])) else L
                    Ea = planned_energy_mJ(model, d, len(sel), 1)
                    if batt - E_step - Ea >= 0:
                        ops = sgd_train(model, Xw[sel], ywo[sel], d, 1, CFG["lr_online"], rng, CFG["momentum"], CFG["batch"])
                        E_step += train_energy_mJ(model, d, ops); n_ad += 1; acts["head" if d == 1 else "full"] += 1
                        n_upd += len(sel)
                    else:
                        acts["skip"] += 1

        elif is_cega:
            fired = any([ph_err.update(e) for e in errs])
            if use_ent: fired = ph_ent.update(H) or fired
            if fired and cool == 0: pending = CFG["pending_max"]
            if pending > 0 and len(buf) >= CFG["min_buffer"]:
                r = CFG["recent"]
                er_n = sum(hist_n[-r:]); er = sum(hist_err[-r:]) / max(er_n, 1)
                ref_n = sum(hist_n[:-r]); ref = sum(hist_err[:-r]) / ref_n if ref_n >= 20 else pre_err
                sev = np.clip((er - ref) / max(ref, 0.05), 0, 2) / 2 if er_n else 0.0
                if use_ent:
                    href = np.mean(hist_ent[:-r]) if len(hist_ent) > r + 2 else pre_ent
                    sev = max(sev, np.clip((np.mean(hist_ent[-r:]) - href) / max(href, 0.05), 0, 2) / 2)
                soc = max(batt / cap, 0.05)
                lam_t = CFG["lambda0"] / (soc if use_soc else 0.5)
                n_tr = int(0.8 * len(buf)); n_va = len(buf) - n_tr
                best_d, best_u = None, 0.0
                for d in depths:
                    Ea = planned_energy_mJ(model, d, n_tr, CFG["max_epochs"] if use_es else CFG["adapt_epochs"], n_va if use_es else 0)
                    u = sev * g[d] - lam_t * (Ea / E_full) * CFG["g_scale"]
                    if batt - Ea >= CFG["reserve"] * cap and u > best_u:
                        best_d, best_u = d, u
                if best_d is not None:
                    Xb, yb = buf_xy()
                    Xt, yt, Xv, yv = Xb[:n_tr], yb[:n_tr], Xb[n_tr:], yb[n_tr:]   # holdout = most recent 20%
                    _, a0 = loss_acc(model, Xv, yv); E_step += n_va * model.macs_fwd() * ENERGY["e_mac_nj"] * 1e-6
                    ops = sgd_train(model, Xt, yt, best_d, CFG["max_epochs"] if use_es else CFG["adapt_epochs"],
                                    CFG["lr_adapt"], rng, CFG["momentum"], CFG["batch"], Xv, yv, early_stop=use_es)
                    _, a1 = loss_acc(model, Xv, yv); E_step += n_va * model.macs_fwd() * ENERGY["e_mac_nj"] * 1e-6
                    E_step += train_energy_mJ(model, best_d, ops)
                    if calib:
                        g[best_d] = (1 - CFG["g_ema"]) * g[best_d] + CFG["g_ema"] * max(a1 - a0, 0.005)
                    n_ad += 1; depth_count[best_d] += 1; events.append((w, best_d))
                    pending = 0; cool = CFG["cooldown"]; ph_err.reset(); ph_ent.reset()
                    hist_err, hist_n, hist_ent = hist_err[-1:], hist_n[-1:], hist_ent[-1:]
                else:
                    pending -= 1

        if E_step > 0:
            E_ad += E_step; batt -= E_step
            if batt < 0:
                viol += 1; deficit += -batt; batt = 0.0
        min_soc = min(min_soc, batt / cap); soc_tr.append(batt / cap)

    y_true = ys[:n]
    res = dict(dataset=ds["name"], method=method, seed=seed, beta=beta,
               acc=float((preds == y_true).mean()),
               macro_f1=float(f1_score(y_true, preds, average="macro")),
               E_inf_mJ=E_inf, E_adapt_mJ=E_ad, E_total_mJ=E_inf + E_ad, n_adapt=n_ad,
               violations=viol, deficit_mJ=deficit, min_soc=min_soc,
               recovery_win=recovery_windows(win_acc, ds["drift"], W),
               d1=depth_count[1], d2=depth_count[2], d3=depth_count[3], E_full_mJ=E_full,
               skip=acts["skip"], head=acts["head"], full=acts["full"], burst=acts["burst"],
               n_labelled=n_lab, n_updated=n_upd, frac_updated=(n_upd / n_lab if n_lab else np.nan),
               p_label=CFG["p_label"], label_noise=CFG.get("label_noise", 0.0), tau_min=CFG["v2_tau_min"],
               tau_max=CFG["v2_tau_max"], e_wr_nj=ENERGY["e_write_nj"], hidden=str(tuple(CFG["hidden"])))
    if keep_trace:
        res["_trace"] = dict(win_acc=win_acc, soc=soc_tr, events=events)
    return res

In [7]:
# ============================== 5. Helpers for experiments ==============================
from contextlib import contextmanager

@contextmanager
def override(cfg=None, energy=None):
    """Temporarily change CFG / ENERGY entries (restored afterwards)."""
    cfg, energy = cfg or {}, energy or {}
    old_c = {k: CFG[k] for k in cfg}; old_e = {k: ENERGY[k] for k in energy}
    CFG.update(cfg); ENERGY.update(energy)
    try:
        yield
    finally:
        CFG.update(old_c); ENERGY.update(old_e)

def run_methods(ds, seed, methods, keep_trace=False):
    """Runs `methods` on one dataset/seed. RandomSkip is matched to CEGA-Tiny's update rate on the same run."""
    rows, out_traces, frac = [], {}, None
    order = [m for m in methods if m != "RandomSkip"]
    need_rs = "RandomSkip" in methods
    if need_rs and "CEGA-Tiny" not in order:
        frac = run_stream(ds, "CEGA-Tiny", seed)["frac_updated"]
    for m in order + (["RandomSkip"] if need_rs else []):
        if m == "RandomSkip" and frac is None:
            frac = [r for r in rows if r["method"] == "CEGA-Tiny"][0]["frac_updated"]
        r = run_stream(ds, m, seed, keep_trace=keep_trace, keep=(frac if m == "RandomSkip" else None))
        tr = r.pop("_trace", None)
        if tr is not None: out_traces[m] = tr
        rows.append(r)
    return rows, out_traces

In [8]:
# ============================== 6. Deployment footprint table ==============================
foot = []
for n in DATASETS:
    ds = get_dataset(n, 0); m, pe, _ = pretrained_model(ds, 0)
    Ef = planned_energy_mJ(m, m.L, CFG["buffer"], CFG["max_epochs"])
    foot.append(dict(dataset=n, params=m.n_params(), model_int8_KB=m.n_params() / 1024,
                     buffer_int8_KB=CFG["buffer"] * ds["Xs"].shape[1] / 1024,
                     MACs_per_inference=m.macs_fwd(),
                     E_inference_uJ=m.macs_fwd() * ENERGY["e_mac_nj"] * 1e-3,
                     E_head_update_mJ=planned_energy_mJ(m, 1, CFG["buffer"], CFG["max_epochs"]),
                     E_full_retrain_mJ=Ef, pretrain_val_acc=1 - pe,
                     n_pre=len(ds["yp"]), n_stream=len(ds["ys"]), windows=len(ds["ys"]) // CFG["window"]))
foot = pd.DataFrame(foot); foot.to_csv(f"{OUT}/tables/footprint.csv", index=False)
print(foot.round(4).to_string(index=False))

  dataset  params  model_int8_KB  buffer_int8_KB  MACs_per_inference  E_inference_uJ  E_head_update_mJ  E_full_retrain_mJ  pretrain_val_acc  n_pre  n_stream  windows
 GasDrift   10534        10.2871         25.0000               10432           2.608            4.2607             9.8315            0.9881   1689     12221      244
  UCI-HAR   38246        37.3496        109.5703               38144           9.536           15.3455            33.5530            0.9724   1934      8365      167
    Elec2    2658         2.5957          1.3672                2560           0.640            1.0533             3.0416            0.8667   4000     41312      826
CoverType    5831         5.6943         10.5469                5728           1.432            2.3937             5.8177            0.8053   5000     55000     1100


In [9]:
# ============================== 7. Main experiment: methods x datasets x seeds ==============================
t0 = time.time(); rows, traces = [], {}
for n in DATASETS:
    for seed in CFG["seeds"]:
        ds = get_dataset(n, seed)
        r, tr = run_methods(ds, seed, ALL_METHODS, keep_trace=(seed == CFG["seeds"][0]))
        rows += r
        for m, t in tr.items(): traces[(n, m)] = t
        print(f"{n:9s} seed {seed} done  ({time.time() - t0:6.1f}s)")
main = pd.DataFrame(rows)
static_acc = main[main.method == "Static"].set_index(["dataset", "seed"]).acc
main["acc_gain_pp"] = 100 * (main.acc - main.set_index(["dataset", "seed"]).index.map(static_acc).values)
main.to_csv(f"{OUT}/raw/main_results.csv", index=False)
print(f"Main experiment finished in {time.time() - t0:.1f}s, {len(main)} runs")

GasDrift  seed 0 done  (   1.7s)
GasDrift  seed 1 done  (   5.1s)
GasDrift  seed 2 done  (   9.0s)
GasDrift  seed 3 done  (  13.2s)
GasDrift  seed 4 done  (  16.5s)
UCI-HAR   seed 0 done  (  18.5s)
UCI-HAR   seed 1 done  (  26.5s)
UCI-HAR   seed 2 done  (  31.5s)
UCI-HAR   seed 3 done  (  39.3s)
UCI-HAR   seed 4 done  (  44.3s)
Elec2     seed 0 done  (  49.7s)
Elec2     seed 1 done  (  55.9s)
Elec2     seed 2 done  (  61.3s)
Elec2     seed 3 done  (  68.0s)
Elec2     seed 4 done  (  73.4s)
CoverType seed 0 done  (  81.1s)
CoverType seed 1 done  (  89.6s)
CoverType seed 2 done  (  98.0s)
CoverType seed 3 done  ( 107.1s)
CoverType seed 4 done  ( 115.3s)
Main experiment finished in 115.4s, 280 runs


In [10]:
# ============================== 8. Sensitivity and robustness sweeps ==============================
def sweep(name, values, methods, make_override):
    t0 = time.time(); out = []
    for n in DATASETS:
        for seed in CFG["sweep_seeds"]:
            ds = get_dataset(n, seed)
            for v in values:
                c, e = make_override(v)
                with override(c, e):
                    r, _ = run_methods(ds, seed, methods)
                for x in r: x.update(sweep=name, value=str(v))
                out += r
    df = pd.DataFrame(out); df.to_csv(f"{OUT}/raw/sweep_{name}.csv", index=False)
    print(f"sweep {name:8s}: {len(df)} runs in {time.time() - t0:.1f}s")
    return df

SW = {}
SW["budget"] = sweep("budget", CFG["beta_sweep"], ["Online-SGD", "Periodic-Full", "DriftTrig-Full", "CEGA-Tiny", "CEGA-Lite"],
                     lambda v: ({"beta": v}, None))
SW["labels"] = sweep("labels", CFG["rho_sweep"], ["Online-SGD", "TinyOL-Head", "DriftTrig-Full", "CEGA-Tiny", "CEGA-Lite"],
                     lambda v: ({"p_label": v}, None))
SW["noise"] = sweep("noise", CFG["noise_sweep"], ["Online-SGD", "RandomSkip", "CEGA-Tiny", "CEGA-Lite"],
                    lambda v: ({"label_noise": v}, None))
TAU = [(a, b) for a in CFG["tau_min_grid"] for b in CFG["tau_max_grid"] if a < b]
SW["tau"] = sweep("tau", TAU, ["CEGA-Tiny"], lambda v: ({"v2_tau_min": v[0], "v2_tau_max": v[1]}, None))
SW["energy"] = sweep("energy", CFG["ewr_sweep"], ["Online-SGD", "TinyOL-Head", "Periodic-Full", "CEGA-Tiny"],
                     lambda v: (None, {"e_write_nj": v}))
SW["model"] = sweep("model", CFG["hidden_sweep"], ["Online-SGD", "TinyOL-Head", "CEGA-Tiny", "CEGA-Lite"],
                    lambda v: ({"hidden": v}, None))

sweep budget  : 300 runs in 100.4s
sweep labels  : 240 runs in 68.6s
sweep noise   : 192 runs in 66.5s
sweep tau     : 144 runs in 49.1s
sweep energy  : 192 runs in 62.9s
sweep model   : 144 runs in 76.0s


In [11]:
# ============================== 9. Statistics ==============================
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for k, i in enumerate(o):
        run = max(run, min(1.0, (len(p) - k) * p[i])); adj[i] = run
    return adj

def compare(df, ref, others, metric):
    piv = df.pivot_table(index=["dataset", "seed"], columns="method", values=metric)
    res = []
    for c in others:
        d = (piv[ref] - piv[c]).dropna()
        if np.allclose(d, 0): st, p = np.nan, 1.0
        else: st, p = stats.wilcoxon(d, zero_method="zsplit")
        ci = stats.t.interval(0.95, len(d) - 1, loc=d.mean(), scale=stats.sem(d)) if d.std() > 0 else (d.mean(), d.mean())
        res.append(dict(metric=metric, comparison=f"{ref} vs {c}", n=len(d), mean_diff=d.mean(), ci_low=ci[0],
                        ci_high=ci[1], d_z=(d.mean() / d.std() if d.std() > 0 else np.nan), W=st, p=p))
    out = pd.DataFrame(res); out["p_holm"] = holm(out.p.values); return out

stat_rows = []
for metric in ["acc", "macro_f1", "E_adapt_mJ"]:
    piv = main[main.method.isin(MAIN_METHODS)].pivot_table(index=["dataset", "seed"], columns="method", values=metric)[MAIN_METHODS]
    fr = stats.friedmanchisquare(*[piv[c].values for c in piv.columns])
    ranks = piv.rank(axis=1, ascending=(metric == "E_adapt_mJ")).mean().sort_values()
    stat_rows.append(pd.DataFrame([dict(metric=metric, comparison="Friedman (all main methods)", n=len(piv), W=fr.statistic,
                                        p=fr.pvalue, mean_diff=np.nan,
                                        note="mean ranks: " + "; ".join(f"{k}={v:.2f}" for k, v in ranks.items()))]))
    stat_rows.append(compare(main, "CEGA-Tiny", [m for m in MAIN_METHODS if m != "CEGA-Tiny"], metric))
    stat_rows.append(compare(main, "CEGA-Lite", ["CEGA-Tiny", "Online-SGD", "RandomSkip"], metric))
stat_df = pd.concat(stat_rows, ignore_index=True); stat_df.to_csv(f"{OUT}/tables/statistics.csv", index=False)
print(stat_df[["metric", "comparison", "mean_diff", "p", "p_holm"]].to_string(index=False))

# Random-skip control on noise sweep (is confidence gating better than random skipping under label noise?)
noise_stats = []
for v, g in SW["noise"].groupby("value"):
    noise_stats.append(compare(g, "CEGA-Tiny", ["RandomSkip", "Online-SGD"], "acc").assign(label_noise=v))
pd.concat(noise_stats).to_csv(f"{OUT}/tables/statistics_noise.csv", index=False)

    metric                  comparison   mean_diff            p   p_holm
       acc Friedman (all main methods)         NaN 6.468440e-21      NaN
       acc         CEGA-Tiny vs Static    0.139592 1.907349e-06 0.000013
       acc     CEGA-Tiny vs Online-SGD    0.003271 6.958008e-02 0.139160
       acc    CEGA-Tiny vs TinyOL-Head    0.063800 9.536743e-06 0.000038
       acc     CEGA-Tiny vs RandomSkip    0.012927 1.049042e-04 0.000315
       acc  CEGA-Tiny vs Periodic-Full    0.001543 5.458755e-01 0.545876
       acc CEGA-Tiny vs DriftTrig-Full    0.028848 1.907349e-06 0.000013
       acc     CEGA-Tiny vs GovRetrain    0.061105 1.907349e-06 0.000013
       acc      CEGA-Lite vs CEGA-Tiny    0.000052 9.404292e-01 0.940429
       acc     CEGA-Lite vs Online-SGD    0.003322 1.446503e-02 0.028930
       acc     CEGA-Lite vs RandomSkip    0.012979 2.670288e-05 0.000080
  macro_f1 Friedman (all main methods)         NaN 3.969616e-20      NaN
  macro_f1         CEGA-Tiny vs Static    0.156559 

In [12]:
# ============================== 10. Tables (CSV + LaTeX) ==============================
def ms(x, k=3):
    return f"{np.nanmean(x):.{k}f} $\\pm$ {np.nanstd(x):.{k}f}" if np.isfinite(np.nanmean(x)) else "--"

def latex_table(df, caption, label, path):
    cols = list(df.columns)
    lines = ["\\begin{table}[t]", "\\centering", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
             "\\resizebox{\\linewidth}{!}{%", "\\begin{tabular}{" + "l" * 2 + "c" * (len(cols) - 2) + "}", "\\toprule",
             " & ".join(str(c) for c in cols) + " \\\\", "\\midrule"]
    prev = None
    for _, r in df.iterrows():
        if prev is not None and r.iloc[0] != prev: lines.append("\\midrule")
        prev = r.iloc[0]
        lines.append(" & ".join(str(v) for v in r.values) + " \\\\")
    lines += ["\\bottomrule", "\\end{tabular}}", "\\end{table}"]
    open(path, "w").write("\n".join(lines))

def summary(df, methods):
    out = []
    for n in DATASETS:
        for m in methods:
            s = df[(df.dataset == n) & (df.method == m)]
            out.append({"Dataset": n, "Method": m, "Accuracy": ms(s.acc), "Macro-F1": ms(s.macro_f1),
                        "Adapt. energy (mJ)": ms(s.E_adapt_mJ, 2), "Updated (\\%)": ms(100 * s.frac_updated, 1),
                        "Violations": ms(s.violations, 1), "Recovery (win.)": ms(s.recovery_win, 1)})
    return pd.DataFrame(out)

t_main = summary(main, MAIN_METHODS + ["CEGA-Lite"]); t_abl = summary(main, ["CEGA-Tiny"] + ABLATIONS)
t_main.to_csv(f"{OUT}/tables/main_table.csv", index=False); t_abl.to_csv(f"{OUT}/tables/ablation_table.csv", index=False)
latex_table(t_main, "Prequential results (mean $\\pm$ std over seeds) at the default energy budget.", "tab:main", f"{OUT}/tables/main_table.tex")
latex_table(t_abl, "Ablation of CEGA-Tiny components.", "tab:ablation", f"{OUT}/tables/ablation_table.tex")
latex_table(foot.round(3), "Deployment footprint and estimated energy (analytic model).", "tab:footprint", f"{OUT}/tables/footprint.tex")
for k, df in SW.items():
    piv = df.pivot_table(index=["dataset", "method"], columns="value", values=["acc", "E_adapt_mJ", "violations"], aggfunc="mean")
    piv.round(4).to_csv(f"{OUT}/tables/sweep_{k}.csv")
print(t_main.to_string(index=False))

  Dataset         Method          Accuracy          Macro-F1 Adapt. energy (mJ)    Updated (\%)     Violations Recovery (win.)
 GasDrift         Static 0.572 $\pm$ 0.029 0.541 $\pm$ 0.037    0.00 $\pm$ 0.00   0.0 $\pm$ 0.0  0.0 $\pm$ 0.0   7.4 $\pm$ 1.6
 GasDrift     Online-SGD 0.834 $\pm$ 0.009 0.826 $\pm$ 0.009   16.77 $\pm$ 0.12 100.0 $\pm$ 0.0  0.0 $\pm$ 0.0  11.0 $\pm$ 1.6
 GasDrift    TinyOL-Head 0.683 $\pm$ 0.039 0.668 $\pm$ 0.041    6.58 $\pm$ 0.05 100.0 $\pm$ 0.0  0.0 $\pm$ 0.0  10.6 $\pm$ 2.7
 GasDrift     RandomSkip 0.813 $\pm$ 0.013 0.805 $\pm$ 0.014    9.33 $\pm$ 0.47  47.7 $\pm$ 3.2  0.0 $\pm$ 0.0   9.9 $\pm$ 3.0
 GasDrift  Periodic-Full 0.862 $\pm$ 0.007 0.856 $\pm$ 0.006   73.68 $\pm$ 0.05   0.0 $\pm$ 0.0  0.4 $\pm$ 0.5   9.9 $\pm$ 3.1
 GasDrift DriftTrig-Full 0.819 $\pm$ 0.023 0.813 $\pm$ 0.022   27.04 $\pm$ 3.01   0.0 $\pm$ 0.0  0.0 $\pm$ 0.0   7.5 $\pm$ 0.7
 GasDrift     GovRetrain 0.762 $\pm$ 0.051 0.754 $\pm$ 0.050   17.32 $\pm$ 2.24   0.0 $\pm$ 0.0  0.0 $\pm$ 0.0 

In [13]:
# ============================== 11. Figures (300 DPI, PNG + PDF) ==============================
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
COL = {"Static": "#7f7f7f", "Online-SGD": "#1f77b4", "TinyOL-Head": "#17becf", "RandomSkip": "#bcbd22",
       "Periodic-Full": "#ff7f0e", "DriftTrig-Full": "#9467bd", "GovRetrain": "#8c564b", "CEGA-Tiny": "#d62728",
       "CEGA-Lite": "#ff9896", "CEGA-noGate": "#e377c2", "CEGA-noBurst": "#393b79", "CEGA-noSoC": "#aec7e8",
       "CEGA-noEntropy": "#98df8a", "CEGA-HeadOnly": "#2ca02c"}

def save(fig, name):
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(f"{OUT}/figures/{name}.{ext}", dpi=300, bbox_inches="tight")
    plt.close(fig)

def smooth(x, k=5):
    x = np.asarray(x, float); w = np.ones(k)
    return np.convolve(x, w, mode="same") / np.convolve(np.ones_like(x), w, mode="same")

ND = len(DATASETS)
# Fig 1: accuracy over the stream
fig, axs = plt.subplots(ND, 1, figsize=(7.2, 2.1 * ND))
for ax, n in zip(np.atleast_1d(axs), DATASETS):
    ds = get_dataset(n, CFG["seeds"][0])
    for m in MAIN_METHODS:
        if (n, m) in traces:
            ax.plot(smooth(traces[(n, m)]["win_acc"]), color=COL[m], lw=1.6 if m == "CEGA-Tiny" else 0.8, label=m)
    for d in ds["drift"]:
        ax.axvline(d // CFG["window"], color="k", ls=":", lw=0.5, alpha=0.5)
    ax.set_title(n); ax.set_ylabel("Window accuracy"); ax.set_ylim(0, 1.02)
np.atleast_1d(axs)[-1].set_xlabel("Stream window"); np.atleast_1d(axs)[0].legend(ncol=4, fontsize=6, loc="lower left")
save(fig, "fig1_accuracy_over_stream")

# Fig 2: energy-accuracy trade-off
fig, axs = plt.subplots(1, ND, figsize=(7.2, 2.5))
for ax, n in zip(np.atleast_1d(axs), DATASETS):
    for m in MAIN_METHODS + ["CEGA-Lite"]:
        s = main[(main.dataset == n) & (main.method == m)]
        ax.errorbar(s.E_adapt_mJ.mean(), s.acc.mean(), xerr=s.E_adapt_mJ.std(), yerr=s.acc.std(),
                    fmt="o", color=COL[m], ms=4, capsize=2, label=m)
    ax.set_xscale("symlog", linthresh=1.0); ax.set_title(n); ax.set_xlabel("Adapt. energy (mJ)")
np.atleast_1d(axs)[0].set_ylabel("Prequential accuracy")
fig.legend(*np.atleast_1d(axs)[0].get_legend_handles_labels(), loc="lower center", ncol=5, fontsize=6, bbox_to_anchor=(0.5, -0.12))
save(fig, "fig2_energy_accuracy_tradeoff")

# Fig 3: battery state of charge
fig, axs = plt.subplots(ND, 1, figsize=(7.2, 1.9 * ND))
for ax, n in zip(np.atleast_1d(axs), DATASETS):
    for m in ["Periodic-Full", "DriftTrig-Full", "Online-SGD", "CEGA-Tiny"]:
        ax.plot(traces[(n, m)]["soc"], color=COL[m], lw=1.1, label=m)
    ax.axhline(CFG["reserve"], color="k", ls="--", lw=0.6); ax.set_title(n); ax.set_ylabel("SoC"); ax.set_ylim(-0.02, 1.02)
np.atleast_1d(axs)[-1].set_xlabel("Stream window"); np.atleast_1d(axs)[0].legend(ncol=4, fontsize=7)
save(fig, "fig3_battery_soc")

def sweep_plot(df, metric_list, fname, xlabel, logx=False, xvals=None):
    methods = list(dict.fromkeys(df.method))
    fig, axs = plt.subplots(len(metric_list), ND, figsize=(7.2, 2.0 * len(metric_list)), squeeze=False)
    for j, n in enumerate(DATASETS):
        for i, (met, lab) in enumerate(metric_list):
            for m in methods:
                g = df[(df.dataset == n) & (df.method == m)].groupby("value")[met].mean()
                order = [str(v) for v in xvals] if xvals is not None else list(g.index)
                x = [float(o) for o in order]
                axs[i, j].plot(x, [g.get(o, np.nan) for o in order], "o-", ms=3, color=COL.get(m, "k"), label=m)
            if logx: axs[i, j].set_xscale("log")
            axs[0, j].set_title(n); axs[i, 0].set_ylabel(lab); axs[-1, j].set_xlabel(xlabel)
    axs[0, 0].legend(fontsize=6)
    save(fig, fname)

sweep_plot(SW["budget"], [("acc", "Accuracy"), ("violations", "Violations")], "fig4_budget_sensitivity",
           r"Energy budget $\beta$", True, CFG["beta_sweep"])
sweep_plot(SW["labels"], [("acc", "Accuracy"), ("E_adapt_mJ", "Adapt. energy (mJ)")], "fig7_label_rate",
           r"Label rate $\rho$", True, CFG["rho_sweep"])
sweep_plot(SW["noise"], [("acc", "Accuracy"), ("E_adapt_mJ", "Adapt. energy (mJ)")], "fig8_label_noise",
           r"Label noise $\eta$", False, CFG["noise_sweep"])
sweep_plot(SW["energy"], [("E_adapt_mJ", "Adapt. energy (mJ)")], "fig10_energy_constants",
           r"$e_{\mathrm{wr}}$ (nJ) with $e_{\mathrm{mac}}=0.25$ nJ", True, CFG["ewr_sweep"])

# Fig 5: ablation
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.8))
abl = ["CEGA-Tiny"] + ABLATIONS; x = np.arange(ND); wdt = 0.8 / len(abl)
for k, m in enumerate(abl):
    a = [main[(main.dataset == n) & (main.method == m)].acc.mean() for n in DATASETS]
    e = [main[(main.dataset == n) & (main.method == m)].E_adapt_mJ.mean() for n in DATASETS]
    axs[0].bar(x + k * wdt, a, wdt, color=COL[m], label=m); axs[1].bar(x + k * wdt, e, wdt, color=COL[m])
for ax, t in zip(axs, ["Accuracy", "Adaptation energy (mJ)"]):
    ax.set_xticks(x + 0.4 - wdt / 2); ax.set_xticklabels(DATASETS, fontsize=7); ax.set_title(t)
lo = main[main.method.isin(abl)].acc.min(); axs[0].set_ylim(max(0, lo - 0.05), None)
fig.legend(*axs[0].get_legend_handles_labels(), loc="lower center", ncol=4, fontsize=6, bbox_to_anchor=(0.5, -0.1))
save(fig, "fig5_ablation")

# Fig 6: governor decisions
fig, ax = plt.subplots(figsize=(4.8, 2.6))
dd = main[main.method == "CEGA-Tiny"].groupby("dataset")[["skip", "head", "full"]].mean().loc[DATASETS]
dd = dd.div(dd.sum(axis=1), axis=0); bottom = np.zeros(ND)
for c, lbl, colr in zip(["skip", "head", "full"], ["Skipped (gated)", "Head-only step", "Full step"], ["#bdbdbd", "#fdae6b", "#a63603"]):
    ax.bar(DATASETS, dd[c].values, bottom=bottom, label=lbl, color=colr, width=0.6); bottom += dd[c].values
ax.set_ylabel("Fraction of labelled windows"); ax.set_ylim(0, 1)
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
save(fig, "fig6_governor_actions")

# Fig 9: gate-threshold heat maps (accuracy and energy relative to default, averaged over datasets)
tau = SW["tau"].copy()
tau["tmin"] = tau.tau_min; tau["tmax"] = tau.tau_max
ref = tau[(tau.tmin == 0.5) & (tau.tmax == 0.95)].groupby(["dataset", "seed"])[["acc", "E_adapt_mJ"]].mean()
tau = tau.join(ref, on=["dataset", "seed"], rsuffix="_ref")
tau["d_acc_pp"] = 100 * (tau.acc - tau.acc_ref); tau["E_rel"] = tau.E_adapt_mJ / tau.E_adapt_mJ_ref
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.6))
for ax, col, ttl, cmap in [(axs[0], "d_acc_pp", "Accuracy change (pp)", "RdBu"), (axs[1], "E_rel", "Energy relative to default", "PuOr_r")]:
    hm = tau.groupby(["tmin", "tmax"])[col].mean().unstack()
    from matplotlib.colors import TwoSlopeNorm
    if col == "d_acc_pp":
        lim = max(0.1, np.nanmax(np.abs(hm.values))); norm = TwoSlopeNorm(vcenter=0.0, vmin=-lim, vmax=lim)
    else:
        norm = TwoSlopeNorm(vcenter=1.0, vmin=min(0.99, np.nanmin(hm.values)), vmax=max(1.01, np.nanmax(hm.values)))
    im = ax.imshow(hm.values, cmap=cmap, aspect="auto", norm=norm)
    ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels(hm.columns); ax.set_yticks(range(hm.shape[0])); ax.set_yticklabels(hm.index)
    ax.set_xlabel(r"$\tau_{\max}$"); ax.set_ylabel(r"$\tau_{\min}$"); ax.set_title(ttl); ax.grid(False)
    for (i, j), v in np.ndenumerate(hm.values):
        if np.isfinite(v): ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="k")
    fig.colorbar(im, ax=ax, fraction=0.046)
save(fig, "fig9_threshold_sensitivity")

# Fig 11: model size
md = SW["model"].copy(); md["params"] = md.value
fig, axs = plt.subplots(2, ND, figsize=(7.2, 4.0), squeeze=False)
for j, n in enumerate(DATASETS):
    for m in md.method.unique():
        g = md[(md.dataset == n) & (md.method == m)].groupby("value")[["acc", "E_adapt_mJ"]].mean()
        order = [str(tuple(h)) for h in CFG["hidden_sweep"]]
        axs[0, j].plot(range(len(order)), [g.acc.get(o, np.nan) for o in order], "o-", ms=3, color=COL[m], label=m)
        axs[1, j].plot(range(len(order)), [g.E_adapt_mJ.get(o, np.nan) for o in order], "o-", ms=3, color=COL[m])
    for i in range(2):
        axs[i, j].set_xticks(range(len(order))); axs[i, j].set_xticklabels([o.replace(" ", "") for o in order], fontsize=6)
    axs[0, j].set_title(n)
axs[0, 0].set_ylabel("Accuracy"); axs[1, 0].set_ylabel("Adapt. energy (mJ)"); axs[0, 0].legend(fontsize=6)
save(fig, "fig11_model_size")
print("Figures saved to", f"{OUT}/figures")

Figures saved to cega_tiny_results/figures


In [14]:
# ============================== 12. OPTIONAL: hardware calibration ==============================
# Run the Arduino sketch `cega_tiny_bench.ino` (supplied with this notebook) on your board, measure the supply
# voltage and the current during each phase with a USB power meter, and fill in HW below. The notebook then
# replaces the assumed energy constants with ones fitted to your board and repeats the Gas Drift comparison.
HW = None
# Example (replace with YOUR measurements):
# HW = dict(board="Arduino Nano 33 BLE", volts=3.3,
#           mA_infer=6.0, us_infer=1500.0,      # one forward pass, 128-64-32-6 MLP
#           mA_full=6.5,  us_full=4800.0)       # one full SGD step on one sample (forward + backward + update)

if HW is not None:
    ds = get_dataset("GasDrift", 0); m, _, _ = pretrained_model(ds, 0)
    E_inf_nj = HW["volts"] * HW["mA_infer"] * 1e-3 * HW["us_infer"] * 1e-6 * 1e9
    E_full_nj = HW["volts"] * HW["mA_full"] * 1e-3 * HW["us_full"] * 1e-6 * 1e9
    e_mac = E_inf_nj / m.macs_fwd()
    e_wr = max((E_full_nj - e_mac * m.macs_train_sample(m.L)) / m.n_params(m.L), 0.0)
    print(f"Calibrated: e_mac = {e_mac:.3f} nJ/MAC, e_wr = {e_wr:.3f} nJ/write (was {ENERGY})")
    hw_rows = []
    with override(None, {"e_mac_nj": e_mac, "e_write_nj": e_wr}):
        for seed in CFG["seeds"]:
            r, _ = run_methods(get_dataset("GasDrift", seed), seed, ["Online-SGD", "TinyOL-Head", "Periodic-Full", "CEGA-Tiny"])
            hw_rows += r
    hw = pd.DataFrame(hw_rows).assign(board=HW["board"], e_mac_nj=e_mac, e_wr_nj_fit=e_wr)
    hw.to_csv(f"{OUT}/raw/hardware_calibrated_gas.csv", index=False)
    print(hw.groupby("method")[["acc", "E_adapt_mJ", "violations"]].mean().round(3))
else:
    print("HW is None: hardware calibration skipped (fill in HW after running the Arduino sketch).")

HW is None: hardware calibration skipped (fill in HW after running the Arduino sketch).


In [15]:
# ============================== 13. Save config & package results ==============================
json.dump(dict(CFG={k: (v if not isinstance(v, dict) else {str(a): b for a, b in v.items()}) for k, v in CFG.items()},
               ENERGY=ENERGY, methods=ALL_METHODS, datasets=DATASETS, smoke=SMOKE, version="v3",
               numpy=np.__version__, python=sys.version.split()[0]),
          open(f"{OUT}/config.json", "w"), indent=2, default=str)
shutil.make_archive(OUT, "zip", OUT)
print("Packaged:", OUT + ".zip")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(OUT + ".zip")

Packaged: cega_tiny_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>